In [1]:
import os
from dotenv import load_dotenv
from groq import Groq

# Load environment variables from .env
load_dotenv()

# Fetch Groq API Key
api_key = os.getenv("GROQ_API_KEY")

if not api_key:
    raise ValueError("❌ GROQ_API_KEY not found in .env file")

# Initialize Groq client
client = Groq(api_key=api_key)

# Use a strong reasoning model
MODEL_NAME = "llama-3.3-70b-versatile"

def get_completion(prompt, model=MODEL_NAME, temperature=0):
    """Helper function to call Groq API"""
    messages = [{"role": "user", "content": prompt}]
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature
    )
    return response.choices[0].message.content

print(f"✅ Setup Complete. Using model: {MODEL_NAME}")

✅ Setup Complete. Using model: llama-3.3-70b-versatile


In [2]:
print("--- ZERO-SHOT PROMPTING ---")

zero_shot_prompt = """
Classify the sentiment of the following text into one of these categories:
[Tech Support, Sales, Refund, Complaint].

Text: "I've been waiting for my money back for 3 weeks now!"
Category:
"""

print("Output:", get_completion(zero_shot_prompt))

--- ZERO-SHOT PROMPTING ---
Output: Category: Refund


In [3]:
print("\n--- FEW-SHOT PROMPTING ---")

few_shot_prompt = """
You are a classification bot.
Answer with ONLY the category name.

Text: "My computer screen is blue."
Category: Tech Support

Text: "I want to buy the new subscription."
Category: Sales

Text: "This service is terrible and I hate it."
Category: Complaint

Text: "I've been waiting for my money back for 3 weeks now!"
Category:
"""

print("Output:", get_completion(few_shot_prompt))


--- FEW-SHOT PROMPTING ---
Output: Complaint


In [4]:
logic_problem = (
    "Roger has 5 tennis balls. "
    "He buys 2 more cans of tennis balls. "
    "Each can has 3 tennis balls. "
    "How many tennis balls does he have now?"
)

print("--- STANDARD PROMPTING ---")
print(get_completion(f"Q: {logic_problem}\nA:"))

print("\n--- CHAIN OF THOUGHT PROMPTING ---")
print(get_completion(f"Q: {logic_problem}\nA: Let's think step by step."))

--- STANDARD PROMPTING ---
To find out how many tennis balls Roger has now, we need to add the tennis balls he already had to the ones he bought.

Roger initially had 5 tennis balls.
He bought 2 cans of tennis balls, with 3 tennis balls in each can. So, he bought 2 * 3 = 6 tennis balls.

Now, we add the tennis balls he already had to the ones he bought: 5 + 6 = 11.

So, Roger has 11 tennis balls now.

--- CHAIN OF THOUGHT PROMPTING ---
To find out how many tennis balls Roger has now, let's break down the information step by step:

1. **Initial Number of Tennis Balls**: Roger starts with 5 tennis balls.

2. **Additional Tennis Balls Purchased**: He buys 2 more cans of tennis balls. Each can contains 3 tennis balls. So, the total number of tennis balls he buys is 2 cans * 3 tennis balls per can = 6 tennis balls.

3. **Total Number of Tennis Balls Now**: To find the total number of tennis balls Roger has now, we add the initial number of tennis balls to the number of tennis balls he purch

In [5]:
class InterviewerBot:
    def __init__(self):
        self.history = [{
            "role": "system",
            "content": (
                "You are an expert Python developer. "
                "Ask clarifying questions before writing any code. "
                "Do not generate code until at least 2 questions are asked."
            )
        }]

    def chat(self, user_input):
        self.history.append({"role": "user", "content": user_input})
        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=self.history,
            temperature=0.7
        )
        reply = response.choices[0].message.content
        self.history.append({"role": "assistant", "content": reply})
        return reply


print("--- INTERVIEW APPROACH DEMO ---")
bot = InterviewerBot()

print("Bot: Hello! Tell me roughly what you want.")

user_inputs = [
    "I want a script to scrape a website.",
    "I want to scrape news headlines from a tech blog.",
    "Just use BeautifulSoup and print them to console."
]

for msg in user_inputs:
    print("\nUser:", msg)
    print("Bot:", bot.chat(msg))

--- INTERVIEW APPROACH DEMO ---
Bot: Hello! Tell me roughly what you want.

User: I want a script to scrape a website.
Bot: Before we dive into writing the script, I have a few questions to clarify your requirements.

1. **What website do you want to scrape?** Is it a specific website, or do you have a list of websites you'd like to scrape? Knowing the website(s) will help me understand the structure and potential challenges of scraping it.

2. **What data do you want to extract from the website?** Are you looking to scrape specific information such as prices, product names, articles, or something else? This will help me determine the best approach for the script.

(I'll ask more questions based on your responses to these initial questions.)

User: I want to scrape news headlines from a tech blog.
Bot: Scraping news headlines from a tech blog can be a useful project.

To further clarify your requirements, I have a few more questions:

3. **Do you have a specific tech blog in mind**, or

In [6]:
def tree_of_thoughts_solver(task):
    print("Task:", task, "\n")

    print("Step 1: Generating reasoning paths...")
    plans = get_completion(
        f"Generate 3 different plans to solve this task:\n{task}"
    )
    print(plans, "\n")

    print("Step 2: Evaluating plans...")
    evaluation = get_completion(
        f"Evaluate the following plans and select the best:\n{plans}"
    )
    print(evaluation, "\n")

    print("Step 3: Executing best plan...")
    final_output = get_completion(
        f"Execute the best plan to solve:\n{task}"
    )
    print(final_output)


print("--- TREE OF THOUGHT DEMO ---")
tree_of_thoughts_solver(
    "Write a one-paragraph sci-fi mystery plot involving time travel."
)

--- TREE OF THOUGHT DEMO ---
Task: Write a one-paragraph sci-fi mystery plot involving time travel. 

Step 1: Generating reasoning paths...
Here are three different plans to solve the task:

**Plan 1: The Chronology Approach**
Start by establishing a clear timeline for the story, including the time period the protagonist originates from and the time period they travel to. Introduce a mystery or anomaly that occurs in the past or future, which the protagonist must investigate and resolve. Develop a plot that involves the protagonist navigating through time, gathering clues, and piecing together the mystery. Finally, craft a thrilling conclusion that reveals the truth behind the mystery and the consequences of the protagonist's actions.

**Plan 2: The Character-Driven Approach**
Begin by creating a compelling protagonist with a personal stake in the time travel mystery. Give them a motivation for traveling through time, such as seeking to prevent a disaster or reunite with a lost loved o